# Multivariate Regression

<img src="./img/4_multivariate-linear-regression.jpeg" width="500px">
<br><br>
<span style="font-size: 70%">Source: <a href="https://xplordat.com/multivariate-linear-regression/">Data Exploration</a></span>
<br><br>

Regressions are not limited to two-dimensional problems.

Sometimes we want to predict a target outcome from several input features, like in the following simple example.

Assume you have records of several programmers fixing different program issues. You data consists of:

<table>
    <tr>
        <th>field name</th>
        <th>data</th>
    </tr>
    <tr>
        <td>dauer</td>
        <td>duration of the fix</td>
    </tr>
    <tr>
        <td>programmierer</td>
        <td>name of the programmer</td>
    </tr>
    <tr>
        <td>bugtyp</td>
        <td>Area of the programming<br>set(GUI, DB, Reporting)</td>
    </tr>
    <tr>
        <td>codelines</td>
        <td>Lines of code of the program<br>(indicating code complexity)</td>
    </tr>
    <tr>
        <td>use cases</td>
        <td>Number of use cases<br>(indicating program complexity)</td>
    </tr>
    <tr>
        <td>alter</td>
        <td>Age of programmer<br>(indicating experience)</td>
    </tr>
</table>
<br><br>

Looking at the data we would like to answer these questions:

- Given a certain size of a program, how long will fixing a software problem take?
- Which features influence problem fixing the most?

Let's try to find out.

__Note__: Due to the limited data available, we will not conduct model verification.

In [ ]:
# import libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
# prepare data
data = pd.read_csv('./data/bugfixes.csv', sep=" ")
X = data[['codelines', 'usecases', 'alter']]
y = data.dauer
data.head()

In [ ]:
data.describe()

In [ ]:
# visualize
sns.set_theme(style='darkgrid')
# sns.set_theme(style='ticks')

df = data[['codelines', 'usecases', 'alter', 'dauer']]
g = sns.pairplot(df, kind='reg')

### Interpretation of the data

Fix duration ranges between 2 and 3.5 hours, a little less than 3 hours on average.  
Our average program length is 355 thousand lines and covers approximately 77 usecases.

Visualization suggests a close relationsship between codelines and usecases (not surprisingly).  
The age of the programmer does not seem to influence the duration of fixes.


In [ ]:
# train the model
lr = LinearRegression()
lr.fit(X, y)

In [ ]:
# model parameters
lr.intercept_, lr.coef_

In [ ]:
# model quality
y_pred_lr = lr.predict(X)
rmse_lr = np.sqrt(mean_squared_error(y, y_pred_lr))
r2_lr = r2_score(y, y_pred_lr)

### Interpretation of the model

In [ ]:
# interpretation

print(f"Minimum time of fix:                  {lr.intercept_:.2f} min.")
print(f"Additional time per 100000 code line: {lr.coef_[0]*100000:.2f} mins.")
print(f"Additional time per use case:         {lr.coef_[1]:.2f} mins.")
print(f"Age dependent penalty:                {lr.coef_[2]:.2f} mins. (starting from age 21y)")

print("\nModel quality:")
print(f"  RMSE:      {rmse_lr:.3f}")
print(f"  R2 score:  {r2_lr:.3f}")


Mathematically:

$t_{fix} = 102.46 + 8.52 \cdot \frac{lines\_of\_code}{100000} + 0.27 \cdot \#\_usecases + 0.45 \cdot \left( age\_of\_developer - 21 \right)$

How long does it take to fix a problem

- in a program with 460.000 lines of code
- with 85 use cases
- by a programmer age 35?

In [ ]:
# predicting duration of fixes

t_min = lr.intercept_
theta_codelines = lr.coef_[0]
theta_usecases = lr.coef_[1]
theta_age = lr.coef_[2]

i_codelines, i_usecases, i_age = 460000, 85, 35

t_fix = t_min + theta_codelines * i_codelines + theta_usecases * i_usecases + theta_age * (i_age - 21)

print(f"Fixing an error in a program with {i_codelines} lines of code / {i_usecases} use cases by a {i_age} y/o programmer will take {t_fix:.2f} minutes")

<img src="./img/0_critical_evaluation.png" width="150px">

### Critical evaluation


- `LinearRegression()` in _Scikit learn_ can handle multivariate regressions

- Alphanumeric fields are not considered (programmer, affected area)

    - including them in the calculation would be a classification problem<br><br>

- The model does not account for correlation between codelines and usecases

    - this would require additional evaluation before fitting the model<br><br>

- The dataset is too small for reliable analysis

- Visualizing multivariate models becomes tricky


<br><br>

<table>
<tr>
<td style="border-style: none"><img src="./img/0_students_input.png" height="100px"></td>
<td style="border-style: none">&nbsp;&nbsp;</td>
<td style="border-style: none; vertical-align: middle"><h5>Students task:</h5>Discuss:
<ul>
    <li>How can we consider cases like codelines / usecases?</li>
    <li>How could alphanumeric values be considered in regression models?</li>
    <li>Which problems can arise from analysing too little data? Give specific examples related to this dataset.</li>
</ul>
</td>
</tr>
</table>